In [18]:
from pathlib import Path
import sys

ROOT = Path.cwd()
DATA_FILE = ROOT / "data" / "wdbc.data"

print("Project folder:", ROOT)
print("Python version:", sys.version.split()[0])
print("Expected dataset path:", DATA_FILE)
print("Dataset found:", DATA_FILE.is_file())

assert DATA_FILE.is_file(), f"Upload wdbc.data to: {DATA_FILE}"

Project folder: C:\Users\Vignesh\Documents\vignesh\BITS\AIML ZG565-ML\Ass_2\A2
Python version: 3.12.1
Expected dataset path: C:\Users\Vignesh\Documents\vignesh\BITS\AIML ZG565-ML\Ass_2\A2\data\wdbc.data
Dataset found: True


In [19]:
%pip install pandas numpy scipy scikit-learn joblib streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import numpy as np
import pandas as pd

# The raw UCI file has no header. The measurements appear in this order.
measurements = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_points", "symmetry",
    "fractal_dimension",
]

# UCI stores ten mean values, ten standard errors, then ten worst values.
features = [
    f"{measurement}_{group}"
    for group in ["mean", "se", "worst"]
    for measurement in measurements
]

raw = pd.read_csv(DATA_FILE, header=None)
assert raw.shape == (569, 32), f"Unexpected raw shape: {raw.shape}"

raw.columns = ["id", "diagnosis"] + features

# Exclude the identifier and diagnosis from the model inputs.
X = raw[features].apply(pd.to_numeric, errors="raise")

# Malignant is the positive class for precision, recall, F1, and AUC.
y = raw["diagnosis"].map({"B": 0, "M": 1})
assert y.notna().all(), "Unexpected or missing diagnosis label."
y = y.astype(int).rename("target")

assert np.isfinite(X.to_numpy()).all(), "Missing or infinite feature value."

print("Input shape:", X.shape)
print("Benign cases:", int((y == 0).sum()))
print("Malignant cases:", int((y == 1).sum()))
display(X.head(3))

Input shape: (569, 30)
Benign cases: 357
Malignant cases: 212


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [21]:
from sklearn.model_selection import train_test_split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

# Save the original, unscaled measurements and their actual labels.
test_data = X_test.copy()
test_data["target"] = y_test
test_data.to_csv(ROOT / "test_data.csv", index=False)

(ROOT / "model").mkdir(exist_ok=True)
(ROOT / "results").mkdir(exist_ok=True)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Saved test CSV shape:", test_data.shape)

Training shape: (455, 30)
Test shape: (114, 30)
Saved test CSV shape: (114, 31)


In [22]:
import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from evaluation import evaluate

# Put scaling inside the pipeline so it is learned from training data.
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED),
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5),
    ),
    "Naive Bayes": make_pipeline(
        StandardScaler(),
        GaussianNB(),
    ),
    "Support Vector Machine": CalibratedClassifierCV(
        estimator=make_pipeline(StandardScaler(), SVC(kernel="rbf")),
        method="sigmoid",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        ensemble=False,
    ),
}

model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib",
}

print("Models ready:", len(models))

Models ready: 5


In [23]:
import json

rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    metrics, _ = evaluate(model, X_test, y_test)
    rows.append({"ML Model Name": name, **metrics})
    joblib.dump(model, ROOT / "model" / model_files[name])

# Keep full precision in the CSV and round only the displayed table.
results = pd.DataFrame(rows).set_index("ML Model Name")
results.to_csv(ROOT / "results" / "metrics.csv")

metadata = {
    "feature_names": features,
    "target_column": "target",
    "target_mapping": {"0": "Benign", "1": "Malignant"},
    "model_files": model_files,
}
(ROOT / "model" / "metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print("BITS ML Assignment 2 — held-out test results")
print(f"Training records: {len(X_train)} | Test records: {len(X_test)}")
display(results.round(4))

BITS ML Assignment 2 — held-out test results
Training records: 455 | Test records: 114


,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
KNN,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058
Naive Bayes,0.9211,0.9891,0.9231,0.8571,0.8889,0.8292
Support Vector Machine,0.9649,0.9947,0.9524,0.9524,0.9524,0.9246


In [24]:
saved_test = pd.read_csv(ROOT / "test_data.csv")

for name, filename in model_files.items():
    saved_model = joblib.load(ROOT / "model" / filename)
    metrics, _ = evaluate(
        saved_model,
        saved_test[features],
        saved_test["target"],
    )
    np.testing.assert_allclose(
        [metrics[column] for column in results.columns],
        results.loc[name].to_numpy(dtype=float),
        atol=1e-8,
        rtol=1e-8,
    )
    print("Verified:", name)

Verified: Logistic Regression
Verified: Decision Tree
Verified: KNN
Verified: Naive Bayes
Verified: Support Vector Machine


In [25]:
from importlib.metadata import version

packages = ["streamlit", "scikit-learn", "numpy", "scipy", "pandas", "joblib"]
requirements = "\n".join(
    f"{package}=={version(package)}" for package in packages
)

(ROOT / "requirements.txt").write_text(
    requirements + "\n", encoding="utf-8"
)

print(requirements)
print("Training Python version:", sys.version.split()[0])

streamlit==1.67.0
scikit-learn==1.9.1
numpy==2.5.4
scipy==1.18.1
pandas==3.1.1
joblib==1.6.0
Training Python version: 3.12.1


In [26]:
print("| ML Model Name | " + " | ".join(results.columns) + " |")
print("|---|" + "---:|" * len(results.columns))

for name, row in results.iterrows():
    values = " | ".join(f"{value:.4f}" for value in row)
    print(f"| {name} | {values} |")

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.9649 | 0.9960 | 0.9750 | 0.9286 | 0.9512 | 0.9245 |
| Decision Tree | 0.9298 | 0.9246 | 0.9048 | 0.9048 | 0.9048 | 0.8492 |
| KNN | 0.9561 | 0.9823 | 0.9744 | 0.9048 | 0.9383 | 0.9058 |
| Naive Bayes | 0.9211 | 0.9891 | 0.9231 | 0.8571 | 0.8889 | 0.8292 |
| Support Vector Machine | 0.9649 | 0.9947 | 0.9524 | 0.9524 | 0.9524 | 0.9246 |
